# Hands-on Workshop: RNNs and LSTMs for Sequence Modeling

**Main task:** Real-world sentiment classification using the IMDB movie review dataset  
**Models covered:** Simple RNN, LSTM, and GRU  
**Platforms supported:** Google Colab, Kaggle Notebook, and local Jupyter Notebook

---

## Learning outcomes

By the end of this notebook, students should be able to:

1. Understand why text is sequential data.
2. Convert raw text into tokens, vocabulary IDs, and padded tensors.
3. Build a basic RNN classifier in PyTorch.
4. Replace RNN with LSTM and understand why LSTM usually works better on long sequences.
5. Compare RNN, LSTM, and GRU on the same real dataset.
6. Test the trained model on custom movie reviews.
7. Understand practical issues: padding, sequence length, batching, GPU usage, overfitting, and inference.

---

This notebook is designed as a **teaching notebook**, not a competition notebook.  
It intentionally avoids advanced NLP tools like Transformers at the start, because the goal is to understand **sequence modeling fundamentals**.

# 1. Platform-safe setup

This notebook should run in:

- **Google Colab**
- **Kaggle Notebook**
- **Local Jupyter Notebook**

Important platform notes:

### Google Colab
Go to:

`Runtime → Change runtime type → GPU`

### Kaggle
Go to:

`Notebook Settings → Accelerator → GPU`

Also enable internet if you want to download the Hugging Face IMDB dataset:

`Notebook Settings → Internet → On`

### Local Jupyter
Install packages manually if needed:

```bash
pip install torch datasets pandas scikit-learn matplotlib tqdm
```

The notebook includes a safe installer cell, but if you are on a restricted server, you may prefer to install packages from your terminal.

In [ ]:
# Safe package check/install for Colab, Kaggle, and local Jupyter.

import sys
import subprocess
import importlib.util


def is_installed(package_name):
    return importlib.util.find_spec(package_name) is not None

required_packages = {
    "torch": "torch",
    "datasets": "datasets",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm"
}

missing = [pip_name for import_name, pip_name in required_packages.items() if not is_installed(import_name)]

if missing:
    print("Missing packages:", missing)
    print("Installing missing packages. This may take a few minutes...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All required packages are already installed.")

In [ ]:
# Imports and reproducibility

import os
import re
import math
import random
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [ ]:
# Platform and GPU detection

def detect_platform():
    if "google.colab" in sys.modules:
        return "Google Colab"
    if os.path.exists("/kaggle/working"):
        return "Kaggle Notebook"
    return "Local Jupyter / Other"

PLATFORM_NAME = detect_platform()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Platform:", PLATFORM_NAME)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 3), "GB")
    print("GPU memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 3), "GB")
else:
    print("No GPU detected. The notebook will still run, but training will be slower.")

# 2. Choose run mode

For classroom teaching, do not train on the full dataset unless you have enough time.

Use:

- `fast`: good for live demo, faster training.
- `high_gpu`: uses more data and larger model.
- `debug`: very small run to test everything quickly.

You can change this before running the dataset and training cells.

In [ ]:
# Change this depending on your teaching situation.

RUN_MODE = "fast"   # options: "debug", "lecture_fast", "high_gpu"

CONFIGS = {
    "debug": {
        "max_train_samples": 1500,
        "max_val_samples": 500,
        "max_test_samples": 1000,
        "max_vocab_size": 8000,
        "max_len": 120,
        "batch_size": 64,
        "embedding_dim": 64,
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.2,
        "epochs": 1,
        "lr": 1e-3
    },
    "fast": {
        "max_train_samples": 12000,
        "max_val_samples": 3000,
        "max_test_samples": 5000,
        "max_vocab_size": 20000,
        "max_len": 220,
        "batch_size": 128,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "num_layers": 1,
        "dropout": 0.3,
        "epochs": 2,
        "lr": 1e-3
    },
    "high_gpu": {
        "max_train_samples": 25000,
        "max_val_samples": 5000,
        "max_test_samples": 25000,
        "max_vocab_size": 40000,
        "max_len": 300,
        "batch_size": 256,
        "embedding_dim": 256,
        "hidden_dim": 256,
        "num_layers": 2,
        "dropout": 0.4,
        "epochs": 4,
        "lr": 1e-3
    }
}

cfg = CONFIGS[RUN_MODE]
cfg

# 3. Real dataset: IMDB movie review sentiment classification

We will use the **IMDB Large Movie Review Dataset**.

Task:

```text
Input:  "This movie was excellent and emotional."
Output: Positive

Input:  "The film was boring and too long."
Output: Negative
```

This is a **many-to-one sequence problem**:

```text
many words  →  one sentiment label
```

We will first try to load from Hugging Face:

```python
load_dataset("stanfordnlp/imdb")
```

If internet is disabled on Kaggle or local Jupyter, the code also checks common Kaggle CSV locations.  
If no real dataset is available, it falls back to a tiny built-in demo dataset only.

In [ ]:
# Load real IMDB dataset with multiple fallbacks.

def load_imdb_from_huggingface():
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb")
    train_df = pd.DataFrame(ds["train"])
    test_df = pd.DataFrame(ds["test"])
    train_df = train_df.rename(columns={"text": "review"})
    test_df = test_df.rename(columns={"text": "review"})
    return train_df[["review", "label"]], test_df[["review", "label"]]


def load_imdb_from_kaggle_csv():
    # Common Kaggle dataset name:
    # /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
    possible_roots = [Path("/kaggle/input"), Path(".")]
    candidate_csvs = []
    for root in possible_roots:
        if root.exists():
            candidate_csvs.extend(list(root.rglob("*.csv")))

    for csv_path in candidate_csvs:
        try:
            sample = pd.read_csv(csv_path, nrows=5)
            cols = {c.lower(): c for c in sample.columns}
            if "review" in cols and "sentiment" in cols:
                df = pd.read_csv(csv_path)
                review_col = cols["review"]
                sentiment_col = cols["sentiment"]
                df = df[[review_col, sentiment_col]].rename(columns={review_col: "review", sentiment_col: "sentiment"})
                df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
                df = df.dropna(subset=["review", "label"])
                df["label"] = df["label"].astype(int)
                df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
                train_df = df.iloc[:40000][["review", "label"]].reset_index(drop=True)
                test_df = df.iloc[40000:][["review", "label"]].reset_index(drop=True)
                if len(test_df) == 0:
                    split = int(0.8 * len(df))
                    train_df = df.iloc[:split][["review", "label"]].reset_index(drop=True)
                    test_df = df.iloc[split:][["review", "label"]].reset_index(drop=True)
                print("Loaded Kaggle/local CSV:", csv_path)
                return train_df, test_df
        except Exception:
            pass

    raise FileNotFoundError("No compatible IMDB CSV found.")


def load_tiny_fallback_dataset():
    print("WARNING: Using tiny fallback dataset. This is only for code testing, not real training.")
    positives = [
        "this movie was excellent and inspiring",
        "i loved the acting and the story",
        "the film was emotional and beautiful",
        "amazing direction and great performances",
        "a wonderful movie with a strong ending",
        "the characters were realistic and memorable",
        "i enjoyed every minute of the film",
        "the story was powerful and moving",
        "brilliant acting and excellent music",
        "one of the best films i have seen"
    ]
    negatives = [
        "this movie was boring and slow",
        "i hated the acting and the story",
        "the film was dull and predictable",
        "bad direction and weak performances",
        "a terrible movie with a poor ending",
        "the characters were flat and forgettable",
        "i did not enjoy this film",
        "the story was weak and confusing",
        "poor acting and annoying music",
        "one of the worst films i have seen"
    ]
    rows = []
    for _ in range(120):
        for text in positives:
            rows.append({"review": text, "label": 1})
        for text in negatives:
            rows.append({"review": text, "label": 0})
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    split = int(0.8 * len(df))
    return df.iloc[:split].reset_index(drop=True), df.iloc[split:].reset_index(drop=True)

try:
    print("Trying Hugging Face dataset: stanfordnlp/imdb")
    train_df, test_df = load_imdb_from_huggingface()
    DATA_SOURCE = "Hugging Face: stanfordnlp/imdb"
except Exception as e1:
    print("Could not load from Hugging Face:", repr(e1))
    try:
        print("Trying Kaggle/local CSV fallback...")
        train_df, test_df = load_imdb_from_kaggle_csv()
        DATA_SOURCE = "Kaggle/local CSV"
    except Exception as e2:
        print("Could not load Kaggle/local CSV:", repr(e2))
        train_df, test_df = load_tiny_fallback_dataset()
        DATA_SOURCE = "Tiny fallback demo"

print("Data source:", DATA_SOURCE)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

In [ ]:
# Create validation split and optionally reduce data for live lecture speed.

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Validation set from training data
val_size = min(cfg["max_val_samples"], max(500, int(0.15 * len(train_df))))
val_df = train_df.iloc[:val_size].reset_index(drop=True)
train_df = train_df.iloc[val_size:].reset_index(drop=True)

# Apply lecture/debug/high_gpu limits
train_df = train_df.iloc[:cfg["max_train_samples"]].reset_index(drop=True)
val_df = val_df.iloc[:cfg["max_val_samples"]].reset_index(drop=True)
test_df = test_df.iloc[:cfg["max_test_samples"]].reset_index(drop=True)

print("Final train:", train_df.shape)
print("Final val:", val_df.shape)
print("Final test:", test_df.shape)

print("\nLabel meaning: 0 = negative, 1 = positive")
print(train_df["label"].value_counts(normalize=True).rename("proportion"))

# 4. Explore the real text data

Before building a model, always look at the data.

- Text reviews have different lengths.
- Neural networks require numerical tensors.
- We need tokenization and padding.
- Long reviews may contain useful information far from the end, which motivates LSTMs.

In [ ]:
# Must run
# Look at examples.

for i in range(3):
    label_name = "positive" if train_df.loc[i, "label"] == 1 else "negative"
    print("=" * 100)
    print("Label:", label_name)
    print(train_df.loc[i, "review"][:1000])

In [ ]:
# Review length analysis

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-z0-9']+", " ", text)
    return text.strip().split()

train_lengths = train_df["review"].apply(lambda x: len(simple_tokenize(x)))

print(train_lengths.describe())

plt.figure(figsize=(8, 4))
plt.hist(train_lengths, bins=50)
plt.title("Distribution of review lengths in tokens")
plt.xlabel("Number of tokens")
plt.ylabel("Number of reviews")
plt.show()

print("Current max_len:", cfg["max_len"])
print("Reviews longer than max_len:", int((train_lengths > cfg["max_len"]).sum()), "out of", len(train_lengths))

# 5. Text preprocessing: tokenization and vocabulary

A neural network cannot understand raw text directly.

We need this conversion:

```text
"this movie was excellent"
        ↓ tokenization
["this", "movie", "was", "excellent"]
        ↓ vocabulary mapping
[12, 45, 18, 309]
        ↓ padding/truncation
[12, 45, 18, 309, 0, 0, 0, ...]
```

Special tokens:

- `<PAD>` = used to make sequences equal length
- `<UNK>` = unknown word not present in vocabulary

In [ ]:
# Build vocabulary from training data only.

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def build_vocab(texts, max_vocab_size=20000, min_freq=2):
    counter = Counter()
    for text in tqdm(texts, desc="Building vocabulary"):
        counter.update(simple_tokenize(text))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for word, freq in counter.most_common(max_vocab_size - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab, counter

vocab, token_counter = build_vocab(
    train_df["review"].tolist(),
    max_vocab_size=cfg["max_vocab_size"],
    min_freq=2
)

id_to_word = {idx: word for word, idx in vocab.items()}

print("Vocabulary size:", len(vocab))
print("Most common tokens:", token_counter.most_common(20))
print("PAD id:", vocab[PAD_TOKEN], "UNK id:", vocab[UNK_TOKEN])

In [ ]:
def encode_text(text, vocab, max_len):
    tokens = simple_tokenize(text)
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens]
    original_len = len(ids)

    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab[PAD_TOKEN]] * (max_len - len(ids))

    return ids, min(original_len, max_len)

sample_text = "This movie was excellent, emotional, and beautifully acted."
sample_ids, sample_len = encode_text(sample_text, vocab, max_len=12)

print("Text:", sample_text)
print("Tokens:", simple_tokenize(sample_text))
print("IDs:", sample_ids)
print("Effective length:", sample_len)

# 6. PyTorch Dataset and DataLoader

The `Dataset` class stores individual examples.

The `DataLoader` creates mini-batches:

```text
batch of reviews → tensor of shape [batch_size, max_len]
batch of labels  → tensor of shape [batch_size]
```

For RNN/LSTM/GRU, we will use `batch_first=True`, so tensor shape is:

```text
[batch_size, sequence_length, embedding_dimension]
```

In [ ]:
# Must run

class IMDBSequenceDataset(Dataset):
    def __init__(self, dataframe, vocab, max_len):
        self.texts = dataframe["review"].tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = encode_text(self.texts[idx], self.vocab, self.max_len)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = IMDBSequenceDataset(train_df, vocab, cfg["max_len"])
val_ds = IMDBSequenceDataset(val_df, vocab, cfg["max_len"])
test_ds = IMDBSequenceDataset(test_df, vocab, cfg["max_len"])

train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print("input_ids shape:", batch["input_ids"].shape)
print("length shape:", batch["length"].shape)
print("label shape:", batch["label"].shape)
print("First label:", batch["label"][0].item())

# 7. Model architecture: RNN / LSTM / GRU classifier

We will use the same classifier structure for all three models:

```text
input word IDs
    ↓
Embedding layer
    ↓
RNN / LSTM / GRU
    ↓
Final hidden state
    ↓
Fully connected layer
    ↓
Positive / Negative prediction
```

Important idea:

- `Embedding` learns vector representation of words.
- `RNN/LSTM/GRU` processes the sequence step by step.
- The final hidden state summarizes the review.
- The classifier predicts sentiment from that summary.

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim=2,
        num_layers=1,
        dropout=0.3,
        model_type="lstm",
        pad_idx=0,
        bidirectional=False
    ):
        super().__init__()
        self.model_type = model_type.lower()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if self.model_type == "rnn":
            self.encoder = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional,
                nonlinearity="tanh"
            )
        elif self.model_type == "lstm":
            self.encoder = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional
            )
        elif self.model_type == "gru":
            self.encoder = nn.GRU(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional
            )
        else:
            raise ValueError("model_type must be one of: rnn, lstm, gru")

        direction_factor = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * direction_factor, output_dim)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)

        if self.model_type == "lstm":
            output, (hidden, cell) = self.encoder(embedded)
        else:
            output, hidden = self.encoder(embedded)

        # hidden shape: [num_layers * num_directions, batch, hidden_dim]
        if self.bidirectional:
            # Last layer forward and backward hidden states
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            final_hidden = hidden[-1]

        final_hidden = self.dropout(final_hidden)
        logits = self.fc(final_hidden)
        return logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


demo_model = SequenceClassifier(
    vocab_size=len(vocab),
    embedding_dim=cfg["embedding_dim"],
    hidden_dim=cfg["hidden_dim"],
    num_layers=cfg["num_layers"],
    dropout=cfg["dropout"],
    model_type="lstm",
    pad_idx=vocab[PAD_TOKEN]
)

print(demo_model)
print("Trainable parameters:", count_parameters(demo_model))

# 8. Training and evaluation utilities

Training loop steps:

1. Send batch to GPU/CPU.
2. Forward pass.
3. Compute loss.
4. Backpropagation.
5. Clip gradients to reduce exploding gradients.
6. Optimizer update.
7. Evaluate on validation set.

Why gradient clipping?

RNN-style models can suffer from exploding gradients.  
Gradient clipping keeps gradients within a safe range.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Training", leave=False)

    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()

        if grad_clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

        progress.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        total_loss += loss.item() * input_ids.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_labels, all_preds


def fit_model(model_type="lstm", bidirectional=False):
    seed_everything(SEED)

    model = SequenceClassifier(
        vocab_size=len(vocab),
        embedding_dim=cfg["embedding_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        model_type=model_type,
        pad_idx=vocab[PAD_TOKEN],
        bidirectional=bidirectional
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

    history = []
    print(f"\nModel: {model_type.upper()} | Bidirectional: {bidirectional}")
    print("Trainable parameters:", count_parameters(model))

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{cfg['epochs']} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

    return model, pd.DataFrame(history)

# 9. Simple RNN

This is the basic recurrent model.


A vanilla RNN can learn short patterns, but it often struggles when the useful word appears far from the prediction point.

Example:

```text
The movie started slowly, but after the interval it became excellent.
```

The model must remember the later positive context correctly.

In [ ]:
rnn_model, rnn_history = fit_model(model_type="rnn", bidirectional=False)
rnn_history

# 10. LSTM

Now we replace the basic RNN with LSTM.

LSTM uses gates to control:

- what to forget,
- what to write,
- what to read/output.

This helps the model preserve useful information over longer sequences.

In [ ]:
lstm_model, lstm_history = fit_model(model_type="lstm", bidirectional=False)
lstm_history

# 11. GRU

GRU is a simpler gated recurrent unit.

It has fewer gates than LSTM and often trains faster.

In [ ]:
# Optional
# Run this if time permits.

RUN_GRU = True

if RUN_GRU:
    gru_model, gru_history = fit_model(model_type="gru", bidirectional=False)
    display(gru_history)
else:
    gru_model, gru_history = None, None
    print("Skipped GRU training.")

# 12. Compare validation performance

In [ ]:
histories = {
    "RNN": rnn_history,
    "LSTM": lstm_history
}

if "gru_history" in globals() and gru_history is not None:
    histories["GRU"] = gru_history

summary_rows = []
for name, hist in histories.items():
    best_row = hist.sort_values("val_acc", ascending=False).iloc[0]
    summary_rows.append({
        "model": name,
        "best_epoch": int(best_row["epoch"]),
        "best_val_acc": float(best_row["val_acc"]),
        "final_train_acc": float(hist.iloc[-1]["train_acc"]),
        "final_val_loss": float(hist.iloc[-1]["val_loss"])
    })

summary_df = pd.DataFrame(summary_rows).sort_values("best_val_acc", ascending=False)
summary_df

In [ ]:
plt.figure(figsize=(8, 4))
for name, hist in histories.items():
    plt.plot(hist["epoch"], hist["val_acc"], marker="o", label=name)

plt.title("Validation accuracy comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 13. Evaluate the best model on test data

Normally, we select the best model using validation data, then report performance on test data.

For simplicity, we will select the model with the highest validation accuracy from our trained models.

In [ ]:
model_objects = {
    "RNN": rnn_model,
    "LSTM": lstm_model
}

if "gru_model" in globals() and gru_model is not None:
    model_objects["GRU"] = gru_model

best_model_name = summary_df.iloc[0]["model"]
best_model = model_objects[best_model_name]

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(best_model, test_loader, criterion, device)

print("Best model:", best_model_name)
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

# 14. Inference: test your own movie reviews

??

In [ ]:
@torch.no_grad()
def predict_sentiment(text, model, vocab, max_len, device):
    model.eval()
    ids, length = encode_text(text, vocab, max_len)
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)

    logits = model(input_ids)
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    pred_label = int(np.argmax(probs))
    pred_name = "positive" if pred_label == 1 else "negative"

    return {
        "text": text,
        "prediction": pred_name,
        "negative_probability": float(probs[0]),
        "positive_probability": float(probs[1])
    }

examples = [
    "The movie started slowly, but the second half was excellent and emotional.",
    "The film had terrible acting and the story was boring.",
    "The visuals were beautiful but the plot was confusing and too long.",
    "I loved the characters, the music, and the powerful ending."
]

for text in examples:
    result = predict_sentiment(text, best_model, vocab, cfg["max_len"], device)
    print("=" * 100)
    print("Review:", result["text"])
    print("Prediction:", result["prediction"])
    print("P(negative):", round(result["negative_probability"], 4))
    print("P(positive):", round(result["positive_probability"], 4))

In [ ]:

my_review = "The movie was slow in the beginning but the ending was powerful and satisfying."

predict_sentiment(my_review, best_model, vocab, cfg["max_len"], device)

# 15. What did the model learn?

The model does not understand English like humans.  
It learns statistical patterns from examples.

For example:

- Words like `excellent`, `amazing`, `wonderful` may push prediction toward positive.
- Words like `boring`, `bad`, `terrible` may push prediction toward negative.
- Longer sentences with contrast words like `but`, `although`, `however` are harder.

Example:

```text
The movie was boring at first, but the ending was excellent.
```

This is a useful example to discuss why sequence order matters.

In [ ]:
# Optional: inspect vocabulary words

interesting_words = ["excellent", "amazing", "boring", "terrible", "but", "not", "good", "bad"]

for word in interesting_words:
    print(f"{word:>12} -> id:", vocab.get(word, "not in vocabulary"))

# 16. Bidirectional LSTM

A bidirectional LSTM reads the sequence in both directions:

```text
forward:   word1 → word2 → word3
backward:  word3 → word2 → word1
```

This often helps in text classification because the model can use both left and right context.

Run this only if you have extra time.

In [ ]:
# Optional advanced cell

RUN_BILSTM = True

if RUN_BILSTM:
    bilstm_model, bilstm_history = fit_model(model_type="lstm", bidirectional=True)
    display(bilstm_history)
else:
    print("Skipped BiLSTM. Set RUN_BILSTM=True to train it.")

# 17. next-word style language modeling intuition

Sentiment classification is a **many-to-one** task.

Language modeling is a **many-to-many** or **many-to-next** task.

Example:

```text
Input:  the movie was very
Target: good
```

In modern systems, Transformers dominate language modeling, but RNN/LSTM language models are still excellent for teaching because they show the idea of predicting the next token from previous tokens.

Below is a small conceptual demo using our vocabulary. This is not a full language model training section, because our main 60-minute lab is sentiment classification.

In [ ]:
# Optional conceptual demo: create next-token pairs from one sentence.

sentence = "the movie was very good and the acting was excellent"
tokens = simple_tokenize(sentence)
ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens]

print("Tokens:", tokens)
print("IDs:", ids)

print("\nNext-token training pairs:")
for i in range(len(tokens) - 1):
    context = tokens[:i+1]
    target = tokens[i+1]
    print(f"Input context: {context}  ->  Target next word: {target}")

# 18. Student exercises

## Exercise 1: Custom reviews
Create five custom movie reviews and check whether the model predicts correctly.

## Exercise 2: Change sequence length
Change:

```python
cfg["max_len"]
```

Try:

- 80
- 150
- 300

Observe the change in speed and accuracy.

## Exercise 3: Change hidden dimension
Change:

```python
cfg["hidden_dim"]
```

Try:

- 64
- 128
- 256

Observe parameter count and accuracy.

## Exercise 4: Compare models
Train:

- RNN
- LSTM
- GRU

Write which model performed best and why.

## Exercise 5: Error analysis
Find examples where the model makes a wrong prediction.

Questions:

- Was the review sarcastic?
- Was it too long?
- Did it contain both positive and negative words?
- Was the sentiment ambiguous?

In [ ]:
# Exercise helper: show some wrong predictions from test set

@torch.no_grad()
def collect_predictions(model, dataframe, max_examples=10):
    model.eval()
    rows = []

    for idx in range(min(len(dataframe), 1000)):
        text = dataframe.loc[idx, "review"]
        true_label = int(dataframe.loc[idx, "label"])
        result = predict_sentiment(text, model, vocab, cfg["max_len"], device)
        pred_label = 1 if result["prediction"] == "positive" else 0

        if pred_label != true_label:
            rows.append({
                "true": "positive" if true_label == 1 else "negative",
                "pred": result["prediction"],
                "p_positive": round(result["positive_probability"], 4),
                "review_start": text[:500]
            })

        if len(rows) >= max_examples:
            break

    return pd.DataFrame(rows)

wrong_df = collect_predictions(best_model, test_df, max_examples=10)
wrong_df

# 19. Save trained model

This section saves:

- model weights,
- vocabulary,
- configuration.

Useful if you want to reuse the model later without retraining.

In [ ]:
# Optional save

SAVE_MODEL = True

if SAVE_MODEL:
    import json

    save_dir = Path("rnn_lstm_workshop_outputs")
    save_dir.mkdir(exist_ok=True)

    model_path = save_dir / f"{best_model_name.lower()}_imdb_model.pt"
    vocab_path = save_dir / "vocab.json"
    config_path = save_dir / "config.json"

    torch.save(best_model.state_dict(), model_path)

    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(vocab, f)

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump({
            "best_model_name": best_model_name,
            "run_mode": RUN_MODE,
            "cfg": cfg,
            "data_source": DATA_SOURCE
        }, f, indent=2)

    print("Saved model to:", model_path)
    print("Saved vocab to:", vocab_path)
    print("Saved config to:", config_path)
else:
    print("SAVE_MODEL=False, nothing saved.")

# 20. Final recap

## What we did

1. Loaded a real-world text classification dataset.
2. Converted text into tokens and numerical IDs.
3. Built a vocabulary.
4. Created padded mini-batches.
5. Trained a Simple RNN.
6. Trained an LSTM.
7. Optionally trained a GRU.
8. Compared model performance.
9. Tested the model on our own examples.



### RNN
RNN reads sequence step by step and updates hidden state.

### LSTM
LSTM is a special RNN with gates.

It controls:

- what to forget,
- what to store,
- what to output.

### GRU
GRU is a simpler gated unit with fewer parameters.

## takeaway

For long text sequences, basic RNNs often struggle.  
LSTM/GRU models are usually stronger because they are designed to preserve useful information across longer time steps.

# 21. References and further reading

- PyTorch `nn.RNN`, `nn.LSTM`, and `nn.GRU` documentation:  
  https://pytorch.org/docs/stable/nn.html

- PyTorch LSTM API:  
  https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html

- Hugging Face IMDB dataset page:  
  https://huggingface.co/datasets/stanfordnlp/imdb

- Kaggle Notebook documentation:  
  https://www.kaggle.com/docs/notebooks

- Original LSTM paper:  
  Hochreiter, S. and Schmidhuber, J. 1997. Long Short-Term Memory. Neural Computation.

---
